# SLAYER-OCR: prywatny audyt najlepszego RF-DETR v1

W Colab wybierz **Runtime > Change runtime type > GPU**, a potem **Runtime > Run all**. Notebook poprosi kolejno o trzy pliki ZIP, sprawdzi ich nazwy i SHA-256, ponownie policzy metryki najlepszego checkpointu na 12 stronach validation, zapisze predykcje oraz porównania GT/model i pobierze jeden prywatny ZIP.

Wynik zawiera obrazy i predykcje. Nie publikuj go na GitHub ani Hugging Face.

In [ ]:
%pip install -q --no-cache-dir "rfdetr[train]==1.11.0"

In [ ]:
import hashlib
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

from google.colab import files
import torch

CODE_REVISION = '9cc606208c6789e799ed42a227f8de037e7caa64'
EXPECTED_HASHES = {
    'dataset': 'f0a157c79756276e9a82e4f8010521190bf72dc8b459c81c3806e840681a0c33',
    'model': '68578a6e008788fd41b6b9f64d31b3a9def82f25c615ead8fc97da5a9962fb0b',
    'evidence': 'ce5e68096866ab903d7722a75da1b0b5cf44215d597da49c7a8137859773fbde',
}
EXPECTED_NAMES = {
    'dataset': 'slayer-layout-rfdetr-private-v1-20260929.zip',
    'model': 'slayer-rfdetr-layout-v1-model.zip',
    'evidence': 'slayer-rfdetr-layout-v1-evidence.zip',
}

assert importlib.metadata.version('rfdetr') == '1.11.0'
assert torch.cuda.is_available(), 'Wybierz środowisko GPU i uruchom notebook od początku.'
print('GPU:', torch.cuda.get_device_name(0))

def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()

def verified_existing(kind):
    path = Path('/content') / EXPECTED_NAMES[kind]
    if not path.exists():
        return None
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == EXPECTED_HASHES[kind], (kind, actual, EXPECTED_HASHES[kind])
    print('VERIFIED', kind, actual)
    return path

def upload_one(kind):
    existing = verified_existing(kind)
    if existing is not None:
        print('REUSED_EXISTING', existing.name)
        return existing
    expected_name = EXPECTED_NAMES[kind]
    print(f'Wybierz {expected_name}. Możesz też zaznaczyć wszystkie trzy ZIP-y naraz.')
    uploaded = files.upload()
    for name, payload in uploaded.items():
        actual = sha256_bytes(payload)
        matched_kind = next((key for key, digest in EXPECTED_HASHES.items() if digest == actual), None)
        if matched_kind is None:
            print('IGNORED_UNKNOWN_FILE', name, actual)
            continue
        canonical = Path('/content') / EXPECTED_NAMES[matched_kind]
        canonical.write_bytes(payload)
        print('CACHED_VERIFIED', matched_kind, canonical.name, actual)
    existing = verified_existing(kind)
    assert existing is not None, (
        f'Nie wgrano wymaganego pliku {expected_name}. Uruchom tę komórkę ponownie i wybierz plik.'
    )
    return existing

## 1/3: dane 48/12
Wybierz `slayer-layout-rfdetr-private-v1-20260929.zip` z katalogu `OCR_engine/data`.

In [ ]:
dataset_archive = upload_one('dataset')

## 2/3: model
Wybierz `slayer-rfdetr-layout-v1-model.zip`.

In [ ]:
model_archive = upload_one('model')

## 3/3: dowody treningowe
Wybierz `slayer-rfdetr-layout-v1-evidence.zip`.

In [ ]:
evidence_archive = upload_one('evidence')

In [ ]:
repo = Path('/content/OCR_engine-audit')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', CODE_REVISION], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', CODE_REVISION], check=True)
resolved = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert resolved == CODE_REVISION, (resolved, CODE_REVISION)
os.chdir(repo)
sys.path.insert(0, str(repo))
print('PINNED_CODE_OK', resolved)

In [ ]:
from training.audit_slayer_rfdetr_layout import run

result_archive, report = run(
    dataset_archive=dataset_archive,
    model_archive=model_archive,
    evidence_archive=evidence_archive,
    expected_hashes=EXPECTED_HASHES,
    output_root='/content',
    code_revision=CODE_REVISION,
    prediction_threshold=0.05,
    audit_threshold=0.25,
)
print('AUDIT_COMPLETE')
print('pages:', report['pages'])
print('best checkpoint mAP 50:95:', report['best_checkpoint_metrics']['val/mAP_50_95'])
print('fixed-threshold counts:', report['fixed_threshold_counts'])
print('hard-example pages:', report['hard_example_pages'])
print('private ZIP:', result_archive)

## Pobranie wyniku
Przeglądarka pobierze `slayer-rfdetr-layout-audit-v1.zip`. Zachowaj ten plik prywatnie i prześlij go do analizy.

In [ ]:
files.download(str(result_archive))